**File** for variables (provided MSA)

In [6]:
PROTEIN_SEQUENCE = "MDSSAAPTNASNCTDALAYSSCSPAPSPGSWVNLSHLDGNLSDPCGPNRTDLGGRDSLCPPTGSPSMITAITIMALYSIVCVVGLFGNFLVMYVIVRYTKMKTATNIYIFNLALADALATSTLPFQSVNYLMGTWPFGTILCKIVISIDYYNMFTSIFTLCTMSVDRYIAVCHPVKALDFRTPRNAKIINVCNWILSSAIGLPVMFMATTKYRQGSIDCTLTFSHPTWYWENLLKICVFIFAFIMPVLIITVCYGLMILRLKSVRMLSGSKEKDRNLRRITRMVLVVVAVFIVCWTPIHIYVIIKALVTIPETTFQTVSWHFCIALGYTNSCLNPVLYAFLDENFKRCFREFCIPTSSNIEQQNSTRIRQNTRDHPSTANTVDRTNHQLENLEAETAPLP"
MSA_FILENAME     = "query.a3m"

LIGANDS = {
    "morphine":   "CN1CC[C@]23[C@@H]4[C@H]1CC5=C2C(=C(C=C5)O)O[C@H]3[C@H](C=C4)O",
    "naloxone":   "C=C[C@H]1CN2CC[C@@]34[C@@H]2[C@H]1CC5=C3C(=C(C=C5)O)O[C@@H]4O",
}

DIFFUSION_SAMPLES = 5
RECYCLING_STEPS   = 10
OUTPUT_DIR        = "/content/output"



Install boltz 2

In [2]:
!pip install requests==2.32.4
!pip install boltz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 7.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.3
    Uninstalling requests-2.32.3:
      Successfully uninstalled requests-2.32.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
boltz 2.2.1 requires requests==2.32.3, but you have requests 2.32.4 which is incompatible.
tsfresh 0.21.1 requires scipy>=1.14.0; python_version >= "3.10", but you have scipy 1.13.1 which is incompatible.
google-adk 1.28.0 requires click<9.0.0,>=8.1.8, but you have click 8.1.7 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
  Using cached requests-2.32.3-py3-none-any.whl.metadata (4.6 kB)
Using cached requests-2.32.3-py3-none-any.whl (64 kB)
  Attempting uninstall: requests
    Found existing installation: requests 2.32

Upload MSA

In [3]:
from google.colab import files

print(f"Please upload: {MSA_FILENAME}")
uploaded = files.upload()

import os
assert MSA_FILENAME in uploaded, f"Expected {MSA_FILENAME}, got {list(uploaded.keys())}"
MSA_PATH = f"/content/{MSA_FILENAME}"
print(f"✓ MSA ready at {MSA_PATH}")

Please upload: query.a3m


Saving query.a3m to query.a3m
✓ MSA ready at /content/query.a3m


Generate YAML files

In [7]:
import os
os.makedirs("/content/yamls", exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

for name, smiles in LIGANDS.items():
    yaml_content = f"""version: 1
sequences:
  - protein:
      id: A
      sequence: {PROTEIN_SEQUENCE}
      msa: {MSA_PATH}
  - ligand:
      id: B
      smiles: "{smiles}"
properties:
  - affinity:
      binder: B
"""
    path = f"/content/yamls/{name}.yaml"
    with open(path, "w") as f:
        f.write(yaml_content)
    print(f"✓ Created {path}")

✓ Created /content/yamls/morphine.yaml
✓ Created /content/yamls/naloxone.yaml


Command

In [8]:
!boltz predict /content/yamls/ --out_dir {OUTPUT_DIR} --diffusion_samples {DIFFUSION_SAMPLES} --recycling_steps {RECYCLING_STEPS}

Checking input data.
Found 0 existing processed inputs, skipping them.
Processing 2 inputs with 2 threads.
100% 2/2 [00:00<00:00,  5.54it/s]
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Running structure prediction for 2 inputs.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/migration/utils.py:56: The loaded checkpoint was produced with Lightning v2.5.0.post0, which is newer than your current Lightning version: v2.5.0
2026-04-03 14:18:46.133245: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775225926.403888    6342 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775225926.479633    6342 cuda_blas.cc:140

Print Results

In [9]:
import json, glob

print(f"\n{'Molecule':<20} {'Binder probability':>20} {'log10(IC50)':>15}")
print("-" * 57)

affinity_files = glob.glob(f"{OUTPUT_DIR}/**/*affinity*.json", recursive=True)

for f in affinity_files:
    name = os.path.basename(f).replace("affinity_", "").replace(".json", "")
    with open(f) as fp:
        data = json.load(fp)
    prob  = data["affinity_probability_binary"]
    affin = data["affinity_pred_value"]
    print(f"{name:<20} {prob:>20.3f} {affin:>15.3f}")


Molecule               Binder probability     log10(IC50)
---------------------------------------------------------
morphine                            0.908          -0.830
naloxone                            0.902          -1.259


Download results

In [10]:
import shutil
shutil.make_archive("/content/boltz_results", "zip", OUTPUT_DIR)
files.download("/content/boltz_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>